In [1]:
import os, sys
from pathlib import Path

cur_dir = os.getcwd()
path = Path(cur_dir)
sys.path.insert(0, str(path.parent.absolute()))

import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from src.preprocess import preprocess_df
from src.network_model import NetworkModel
from src.analyze_cic_ids import nre_classification, flow_based_classification
from src.classification_tools import plot_roc_curves

from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB


In [2]:
file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Tuesday-WorkingHours.pcap_ISCX.csv'  # 'Monday-WorkingHours.pcap_ISCX.csv' #  Wednesday-workingHours.pcap_ISCX.csv
#file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv'
#file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv'
#file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Friday-WorkingHours-Morning.pcap_ISCX.csv'
#file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv'
#file_addr = '..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'
df_cic = pd.read_csv(file_addr, header=0, encoding='cp1252')

df = preprocess_df(df_cic, date_col=' Timestamp')
print(df.shape)

(445645, 85)


In [3]:
with open('victim_net.pickle', 'rb') as handle:
    entity_names = pickle.load(handle) 
len(entity_names)

12

In [4]:
import importlib
import src.validation_tools

importlib.reload(src.validation_tools)
from src.validation_tools import *

In [5]:
import importlib
import src.analyze_cic_ids

importlib.reload(src.analyze_cic_ids)
from src.analyze_cic_ids import *

In [6]:
def model_nre(df_train, ml_models, **kwargs):
    return nre_classification(df_train, ml_models, entity_names=entity_names, verbose=False, **kwargs)

def model_fb(df_train, ml_models, **kwargs):
    return flow_based_classification(df_train, ml_models, entity_names=entity_names, **kwargs)

val_ratio, test_ratio = 0.25, 0.25
n_all = df_cic.shape[0]
n_train = int(n_all * (1 - val_ratio - test_ratio))
n_val = int(n_all * val_ratio)

df_train = df.iloc[:n_train, :]
df_val = df.iloc[n_train:n_train + n_val, :]
df_test = df.iloc[n_train + n_val:, :]

param_list = [{'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 90, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 45, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 360, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 720, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
               {'t_graph': 180, 'sync_window_size': 0.6, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
               {'t_graph': 180, 'sync_window_size': 0.3, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
               {'t_graph': 180, 'sync_window_size': 2.4, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
               {'t_graph': 180, 'sync_window_size': 4.8, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.2, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.8, 'relief_factor': 0.6, 'k_steps': 1},
              {'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 3},
              {'t_graph': 180, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 10}
              ]
auc_scores_nre = validate_model(df_train, df_val, model_nre, param_list)
auc_scores_fb = validate_model(df_train, df_val, model_fb, param_list)

  0%|                                                                                                                         | 0/13 [00:00<?, ?it/s]C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
  8%|████████▋                                                                                                        | 1/13 [01:35<19:01, 95.11s/it]C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
 23%|██████████████████████████                                                   

 62%|█████████████████████████████████████████████████████████████████████▌                                           | 8/13 [03:11<01:49, 21.91s/it]C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
 69%|██████████████████████████████████████████████████████████████████████████████▏                                  | 9/13 [03:33<01:27, 21.86s/it]C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
 77%|█████████████████████████████████████████████████████████████████████████████

In [9]:
res = (auc_scores_nre, auc_scores_fb)
with open('validation_curves1.pickle', 'wb') as handle:
    pickle.dump(res, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [10]:
with open('validation_curves1.pickle', 'rb') as handle:
    auc_scores_nre, auc_scores_fb = pickle.load(handle) 

In [11]:
auc_scores_nre

[({'t_graph': 180,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.9523809523809524),
 ({'t_graph': 90,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.9375),
 ({'t_graph': 45,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.971086739780658),
 ({'t_graph': 360,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.8636363636363636),
 ({'t_graph': 720,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  1.0),
 ({'t_graph': 180,
   'sync_window_size': 0.6,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.9523809523809524),
 ({'t_graph': 180,
   'sync_window_size': 0.3,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.9523809523809524),
 ({'t_graph': 180,
   'sync_window_size': 2.4,
   'forget_factor

In [12]:
auc_scores_fb

[({'t_graph': 180,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.7142857142857143),
 ({'t_graph': 90,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.65),
 ({'t_graph': 45,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.576271186440678),
 ({'t_graph': 360,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.8181818181818181),
 ({'t_graph': 720,
   'sync_window_size': 1.2,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.6666666666666666),
 ({'t_graph': 180,
   'sync_window_size': 0.6,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.7142857142857143),
 ({'t_graph': 180,
   'sync_window_size': 0.3,
   'forget_factor': 0.5,
   'relief_factor': 0.6,
   'k_steps': 1},
  0.7142857142857143),
 ({'t_graph': 180,
   'sync_window_size': 2.4,
   '

In [9]:
param_list = [{'t_graph': 90, 'sync_window_size': 1.2, 'forget_factor': 0.5, 'relief_factor': 0.6, 'k_steps': 1}]
auc_scores_nre = validate_model(df_train, df_val, model_nre, param_list)

  0%|                                                                                                                          | 0/1 [00:00<?, ?it/s]C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\sklearn\utils\_array_api.py:380: RuntimeWarning: overflow encountered in cast
  array = numpy.asarray(array, order=order, dtype=dtype)
  0%|                                                                                                                          | 0/1 [01:18<?, ?it/s]


ValueError: Input X contains infinity or a value too large for dtype('float32').